# Tech Challenge - Fase 1 (IADT)

## Classificação de câncer de mama

Este é o notebook principal da análise com **dados estruturados** do Breast Cancer Wisconsin. A proposta aqui é apresentar, de forma clara e organizada, o fluxo obrigatório do trabalho: entender o problema, explorar a base, preparar os dados, treinar os modelos e interpretar os resultados.

Ao longo do notebook, seguimos este percurso:

- contexto do problema e da base escolhida
- carregamento e exploração dos dados
- pré-processamento
- treinamento de diferentes modelos
- avaliação com accuracy, recall, F1-score, ROC e matrizes de confusão
- explicabilidade
- ajuste de limiar com foco clínico

Os estudos complementares com radiômica, CNN e multimodalidade ficam separados no notebook `02_estudo_integrado.ipynb`, para preservar aqui o núcleo principal da entrega.


## 1. Preparação do ambiente

Nesta etapa, adicionamos o diretório `src/` ao caminho de busca do Python. Com isso, o pacote do projeto pode ser importado diretamente no notebook, sem exigir uma instalação prévia no ambiente.


In [ ]:
%matplotlib inline
import os, sys
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve, roc_auc_score
from sklearn.model_selection import cross_validate

from techchallenge import config
from techchallenge.data import estruturado as dados
from techchallenge.models import estruturado as modelos_estr
from techchallenge.evaluation import metrics, explicabilidade, limiar

sns.set_theme(style="whitegrid")
OUT = config.garantir_outputs()
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
config.PROJETO_DIR, OUT

## 2. Problema e base escolhida

O objetivo é classificar tumores de mama em **malignos** ou **benignos** a partir de características numéricas extraídas dos exames. Para essa etapa do trabalho, utilizamos o **Breast Cancer Wisconsin Dataset**, uma base bastante conhecida em tarefas de classificação supervisionada, com 569 registros e 30 variáveis preditoras.

Antes da análise, a função `carregar_dados()` faz uma limpeza inicial na base. Nesse processo, as colunas `id` e `Unnamed: 32` são removidas: a primeira é apenas um identificador de registro, enquanto a segunda está vazia e não acrescenta informação útil ao modelo.


In [ ]:
df = dados.carregar_dados()
print("Formato:", df.shape)
df.head()

## 3. Análise exploratória dos dados (EDA)

Antes de treinar os modelos, vale entender como a base se comporta. Nesta etapa, observamos a distribuição da variável-alvo, as estatísticas descritivas e as correlações mais associadas à malignidade.

Essa leitura inicial ajuda a contextualizar o problema, identificar possíveis padrões e levantar hipóteses sobre quais variáveis tendem a ter maior peso na classificação.


In [ ]:
counts = df["diagnosis"].value_counts().rename(index={"B": "Benigno", "M": "Maligno"})
display(counts.to_frame("quantidade"))

plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="diagnosis", order=["B", "M"], palette=["#4c9f70", "#d1495b"])
plt.title("Distribuicao do diagnostico")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.show()

In [ ]:
df.describe().T.head(10)

In [ ]:
df_corr = df.copy()
df_corr["diagnosis"] = df_corr["diagnosis"].map({"M": 1, "B": 0})
corr = df_corr.corr(numeric_only=True)["diagnosis"].drop("diagnosis").sort_values(ascending=False)
print("Top 10 features mais correlacionadas com malignidade:")
display(corr.head(10).round(3).to_frame("correlacao"))

plt.figure(figsize=(8, 10))
corr.head(15).sort_values().plot(kind="barh", color="#d1495b")
plt.title("Top correlacoes com malignidade")
plt.xlabel("Correlacao de Pearson")
plt.show()

In [ ]:
top_cols = corr.head(10).index.tolist()
plt.figure(figsize=(10, 8))
sns.heatmap(df_corr[top_cols + ["diagnosis"]].corr(), cmap="coolwarm", center=0)
plt.title("Heatmap das principais features")
plt.show()

## 4. Pré-processamento

Como a base possui mais casos benignos do que malignos, utilizamos uma divisão **estratificada** entre treino e teste. Isso garante que a proporção entre as classes seja preservada nos dois conjuntos e torna a avaliação mais representativa.

Além disso, o pacote do projeto aplica internamente as seguintes etapas:

- remoção de colunas não informativas
- codificação da variável-alvo (`M = 1` e `B = 0`)
- separação entre treino e teste com estratificação
- padronização dentro de `Pipeline`, evitando vazamento de dados

Esse cuidado é importante para manter o fluxo consistente entre os modelos e assegurar que a comparação seja justa.


In [ ]:
X_train, X_test, y_train, y_test = dados.preparar_treino_teste(df)
print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("Proporcao no treino:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Proporcao no teste :", y_test.value_counts(normalize=True).round(3).to_dict())

## 5. Modelagem

Nesta etapa, treinamos os **seis modelos de base** do projeto: Regressão Logística, Árvore de Decisão, KNN, Random Forest, Gradient Boosting e SVM. Cada modelo é organizado dentro de um *pipeline*, o que mantém o pré-processamento acoplado ao classificador e reduz o risco de inconsistências entre treino e teste.

Além dos modelos individuais, avaliamos também uma estratégia de **Stacking**. Nessa abordagem, os modelos de base geram previsões no primeiro nível, e um metamodelo de Regressão Logística aprende a combinar esses sinais para produzir a decisão final.

Na prática, essa combinação busca aproveitar forças diferentes de cada técnica. Para evitar vazamento de informação, o Stacking utiliza validação cruzada interna antes do ajuste final do metamodelo.


In [ ]:
modelos = modelos_estr.construir_modelos()
modelos["Stacking"] = modelos_estr.construir_stacking(modelos_estr.construir_modelos())

resultados = []
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    linha = {"Modelo": nome, **metrics.metricas(y_test, y_pred)}
    cv = cross_validate(modelo, X_train, y_train, cv=5, scoring=["accuracy", "recall"], n_jobs=-1)
    linha["CV Acc (media)"] = cv["test_accuracy"].mean()
    linha["CV Recall (media)"] = cv["test_recall"].mean()
    resultados.append(linha)

res_df = pd.DataFrame(resultados).set_index("Modelo").round(4).sort_values(
    by=["Recall (maligno)", "Accuracy"], ascending=False
)
res_df

## 6. Avaliação

Na comparação entre os modelos, a métrica de maior interesse é o **recall da classe maligna**, porque o erro mais crítico neste contexto é o falso negativo - ou seja, quando um caso maligno é classificado como benigno.

Ainda assim, o recall não é analisado isoladamente. Também consideramos **accuracy**, **F1-score**, **ROC/AUC** e as **matrizes de confusão**, para manter uma visão mais equilibrada do desempenho de cada abordagem. Todos os modelos são avaliados sobre o mesmo conjunto de teste.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.ravel()

for ax, (nome, modelo) in zip(axes, modelos.items()):
    cm = confusion_matrix(y_test, modelo.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=["Benigno", "Maligno"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(nome)

for ax in axes[len(modelos):]:
    ax.axis("off")

plt.suptitle("Matrizes de confusao - modelos do trabalho", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))
for nome, modelo in modelos.items():
    if hasattr(modelo, "predict_proba"):
        prob = modelo.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc = roc_auc_score(y_test, prob)
        lw = 2.5 if nome == "Stacking" else 1.3
        plt.plot(fpr, tpr, lw=lw, label=f"{nome} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("Falso positivo (1 - especificidade)")
plt.ylabel("Verdadeiro positivo (recall)")
plt.title("Curvas ROC - modulo estruturado")
plt.legend(loc="lower right", fontsize=8)
plt.show()

## 7. Explicabilidade

Como o Stacking combina múltiplos modelos, sua interpretação direta se torna menos transparente. Por isso, complementamos a análise com duas visões mais explicáveis: os coeficientes da Regressão Logística e os valores SHAP da Árvore de Decisão.

Esses recursos ajudam a identificar quais características mais influenciam as previsões e tornam a discussão dos resultados mais sólida do ponto de vista analítico.


In [ ]:
explicabilidade.feature_importance_logistica(
    modelos["Regressao Logistica"],
    X_train.columns,
    OUT / "estruturado_feature_importance.png",
)
display(Image(str(OUT / "estruturado_feature_importance.png")))

In [ ]:
explicabilidade.shap_arvore(
    modelos["Arvore de Decisao"],
    X_test,
    X_train.columns,
    OUT / "estruturado_shap.png",
)
display(Image(str(OUT / "estruturado_shap.png")))

## 8. Ajuste de limiar com foco clínico

Depois de identificar o melhor modelo global, analisamos o efeito de variar o limiar de decisão. Esse passo é importante porque, em cenários clínicos, muitas vezes faz sentido aceitar mais falsos positivos para reduzir a chance de falsos negativos.

Aqui, o foco é entender esse equilíbrio e verificar em que ponto o modelo passa a favorecer mais o **recall** da classe maligna.


In [ ]:
prob = modelos["Stacking"].predict_proba(X_test)[:, 1]
tab = limiar.tabela_limiares(y_test, prob)
t_recall = limiar.escolher_limiar_por_recall(y_test, prob, recall_alvo=1.0)
limiar.plot_trade_off(tab, OUT / "estruturado_limiar_tradeoff.png", t_recall)
limiar.comparar_matrizes(y_test, prob, 0.5, t_recall, OUT / "estruturado_limiar_matrizes.png")

def resumo_limiar(t):
    pred = (prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    rec = tp / (tp + fn)
    return {"limiar": t, "FN": int(fn), "FP": int(fp), "Recall": round(rec, 4)}

pd.DataFrame([resumo_limiar(0.5), resumo_limiar(t_recall)])

In [ ]:
display(Image(str(OUT / "estruturado_limiar_tradeoff.png")))
display(Image(str(OUT / "estruturado_limiar_matrizes.png")))

## 9. Conclusão

Este notebook consolida a parte principal do trabalho com dados estruturados e mostra, de ponta a ponta, como o problema foi tratado: da leitura da base até a interpretação dos resultados.

De forma geral, o **Stacking** aparece como a alternativa mais forte em desempenho global, enquanto a **Regressão Logística** continua sendo uma referência importante em interpretabilidade. Em um contexto aplicado, esse equilíbrio entre desempenho e capacidade de explicação é especialmente valioso.

Os estudos complementares - especialmente os relacionados a imagens, radiômica e estratégias multimodais - ficam organizados separadamente no notebook `02_estudo_integrado.ipynb`.
